Publication Visuals and Analytics - BC
-----

# ----- [ LANDS ] ---------

## Define RUN TAG

In [ ]:
weather_year='2024'
run_date='20260603'
scenario_name='BASELINE'

- Tested tweaks on Plot Adjustments

In [ ]:
MARKER_SCALE_EXISTING:float=1.5
MARKER_SCALE_COMMITTED:float=0.4
MARKER_HIGHLIGHT_WIDTH:float=3
ANNOTATION_ADJUSTMENT_METER:int=100E3 

- Plot and data aggregation

In [ ]:
AGGREGATION_LEVEL='Province'

- Set Root

In [ ]:
from pathlib import Path
current_dir=Path.cwd()
# Go up 2 folders
root = current_dir.parent.parent
print('Root:', root)

paper_resources=current_dir
print('Paper Resources:', paper_resources)

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import RESource.visuals as vis
from RESource import utility as utils
from RESource.hdf5_handler import DataHandler
from RESource.CellCapacityProcessor import get_sub_nationally_aggregated_capacity
plt.style.use(root / 'RES' / 'visual_styles' / 'elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)

- Load Configs

In [ ]:
RUN_ID = f'{scenario_name.upper()}_{weather_year}_{run_date}'
config_name=f'CAN_{scenario_name.lower()}.yaml'

In [ ]:
cfg=utils.load_config(root / 'config' / config_name)
# run_id:str=cfg.get('Scenario').get('run_id')
country_name:str=cfg.get('country')
country_kwd=country_name.replace(' ','')
region_code='BC'
region_name=cfg.get('region_mapping').get(region_code).get('name')

In [ ]:
cfg.get('region_mapping').get(region_code).get('CRS_meters')

In [ ]:
sub_national_unit_tag:str=cfg.get('GADM').get('datafield_mapping').get('NAME_2',None) or cfg.get('GADM').get('datafield_mapping').get('NAME_1')

CRS_m = cfg.get('region_mapping').get(region_code).get('CRS_meters') or cfg.get('default_CRS').get('meters')  # Default metric CRS
CRS_d = cfg.get('default_CRS').get('degrees')  # Default geographic CRS
# regions=list(cfg.get('region_mapping').keys()) #'AL','BA','XK','ME','MK','RS'
regions=['BC']
region_to_code = {
    v["name"].replace(" ", ""): k
    for k, v in cfg.get("region_mapping").items()
}

vis_save_to_root=utils.ensure_path(paper_resources / "vis/Lands")
results_save_to_root=utils.ensure_path(paper_resources / "results")

In [ ]:
plot_crs=CRS_m

## Load Store

In [ ]:
 #All the regions should have RUN_ID results available
combined_store:dict[dict] = {}
utils.print_update(level=1,message=f"Loading data stores for {country_name} regions with RUN_ID: {RUN_ID} and Regions: {regions}")
for region in regions:
    country_dict = {}
    try:
        store = Path(root / f"data/store/{country_kwd}/{region_code}/resources_{country_kwd}_{region}_{RUN_ID}.h5")
        assert store.exists(), f"Store path doesn't exist: {store}"
        res_data = DataHandler(store, show_structure=False)
        country_dict['cells'] = res_data.from_store('cells')
        country_dict['boundary'] = res_data.from_store('boundary')
        country_dict['lines'] = res_data.from_store('lines')
        country_dict['timeseries_solar'] = res_data.from_store('timeseries/solar')
        country_dict['timeseries_wind'] = res_data.from_store('timeseries/wind')
        # country_dict['di_solar'] = res_data.from_store('dissolved_indices/solar')
        # country_dict['di_wind'] = res_data.from_store('dissolved_indices/wind')                                                                 
        # country_dict['LandAvailability'] = res_data.from_store('LandAvailability')
        combined_store[region] = country_dict
        utils.print_update(level=2,message=f"✓ Loaded data for : {cfg.get('region_mapping').get(region).get('name') if cfg.get('region_mapping').get(region) else region}.") 
    except Exception as e:
        print(f"X Error with region {region}: {e}")
        continue

## Load All Cells

In [ ]:
all_cells_dict:dict[pd.DataFrame]=[combined_store[region]['cells'] for region in combined_store]
all_cells_gdf = gpd.GeoDataFrame(pd.concat(all_cells_dict, ignore_index=False), crs=all_cells_dict[0].crs)
all_cells_gdf['ISO2']=all_cells_gdf['Province'].apply(lambda x: region_to_code[x.replace(" ","")])

- Create cells' instance for plotting (CRS-m)

In [ ]:
if all_cells_gdf.crs != plot_crs:
    all_cells_gdf_plot = all_cells_gdf.to_crs(plot_crs)
else:
    all_cells_gdf_plot = all_cells_gdf

# Load Validation data 

## Existing VRE sites

In [ ]:
existing_VREs_data_path=Path(root/f"data/downloaded_data/CODERS/data-pull/supply/{region_code}_wind_generators.csv")
if existing_VREs_data_path.exists():
    existing_VREs=pd.read_csv(existing_VREs_data_path)
    existing_VREs_gdf=gpd.GeoDataFrame(existing_VREs,geometry=gpd.points_from_xy(existing_VREs.longitude,existing_VREs.latitude),crs=CRS_d)
    utils.print_update(level=1,message=f"Validation data for existing VREs loaded from {existing_VREs_data_path}")
    
    existing_tech_name_mapping={
    'wind_ons':'Wind',
    'solar':'Solar'
    }
    # Create new column 'Technology' based on mapping
    existing_VREs_gdf["Technology"] = existing_VREs_gdf["gen_type"].map(existing_tech_name_mapping)

    # If some gen_type values are not in the dict, fill them with 'Unknown'
    existing_VREs_gdf["Technology"] = existing_VREs_gdf["Technology"].fillna("Unknown")

else:
    existing_VREs_gdf=None
    utils.print_warning(f"Validation data for existing VREs not found at {existing_VREs_data_path}")


if existing_VREs_gdf.crs != plot_crs:
    existing_VREs_plot = existing_VREs_gdf.to_crs(plot_crs)
else:
    existing_VREs_plot = existing_VREs_gdf

## Committed VRE Sites (BCH CFPs)

In [ ]:
committed_VREs_data_path=Path(root/"ROD_2024/BCH_CFP24.geojson")
if committed_VREs_data_path.exists():
    committed_VREs_gdf=gpd.read_file(committed_VREs_data_path, engine="fiona")
    if committed_VREs_gdf.crs is None:
        committed_VREs_gdf.set_crs(CRS_d, allow_override=True, inplace=True)
    # if committed_VREs_gdf.crs != CRS_m:
    #     committed_VREs_gdf.to_crs(CRS_m, inplace=True)
    utils.print_update(level=1,message=f"Committed VREs data loaded from {committed_VREs_data_path}")
    
    
    committed_tech_name_mapping={
        'wind':'Wind',
        'solar':'Solar'
    }
    # Create new column 'Technology' based on mapping
    committed_VREs_gdf["Technology"] = committed_VREs_gdf["resource_type"].map(existing_tech_name_mapping)

    # If some gen_type values are not in the dict, fill them with 'Unknown'
    committed_VREs_gdf["Technology"] = committed_VREs_gdf["resource_type"].fillna("Unknown")


else:
    committed_VREs_gdf=None
    utils.print_warning(f"Validation data for Committed VREs not found at {committed_VREs_data_path}")


if committed_VREs_gdf.crs != plot_crs:
    committed_VREs_plot = committed_VREs_gdf.to_crs(plot_crs)
else:
    committed_VREs_plot = committed_VREs_gdf

# Boundary

- Prepare regional boundary

In [ ]:
# Combine all region boundaries into a single GeoDataFrame
boundary_gdfs = [combined_store[region]['boundary'] for region in combined_store]
all_regions_boundary = gpd.GeoDataFrame(pd.concat(boundary_gdfs, ignore_index=True), crs=boundary_gdfs[0].crs)
all_regions_boundary_dissolved = all_regions_boundary.dissolve(by=AGGREGATION_LEVEL)[["geometry"]].reset_index()

- Process the boundary info for raster plotting

In [ ]:
all_regions_boundary_dissolved_plot=all_regions_boundary_dissolved.to_crs(CRS_m)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = all_regions_boundary_dissolved_plot.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

# Lands

- Prepare Combined Regional boundary

In [ ]:
# Combine all region boundaries into a single GeoDataFrame
boundary_gdfs = [combined_store[region]['boundary'] for region in combined_store]
all_boundary_gdf = gpd.GeoDataFrame(pd.concat(boundary_gdfs, ignore_index=True), crs=boundary_gdfs[0].crs)
all_boundary_dissolved = all_boundary_gdf.dissolve(by=AGGREGATION_LEVEL)[["geometry"]].reset_index()

In [ ]:
boundary_save_to_path = Path(paper_resources/"data/boundary_BC.geojson")
all_boundary_dissolved.to_file(boundary_save_to_path, driver="GeoJSON")

- Process the boundary info for raster plotting

In [ ]:
all_boundary_plot=all_boundary_dissolved.to_crs(plot_crs)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = all_boundary_plot.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

* Country Map (base)

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.patheffects as pe

# fig, ax = plt.subplots(figsize=(6, 6),dpi=500)
# fig.suptitle(region_name, fontsize=12, fontweight='bold')

# all_boundary_plot.plot(
#     column=AGGREGATION_LEVEL,
#     categorical=True,
#     edgecolor="black",
#     linewidth=0.2,
#     alpha=1,
#     ax=ax
# )

# for _, row in all_boundary_plot.iterrows():
#     point = row.geometry.representative_point()
#     txt = ax.annotate(
#         row[AGGREGATION_LEVEL],
#         xy=(point.x, point.y),
#         ha="center",
#         va="center",
#         fontsize=10,
#         fontweight="bold",
#         color="black"
#     )
#     # txt.set_path_effects([
#     #     pe.withStroke(linewidth=0.1, foreground="white")
#     # ])

# ax.set_axis_off()
# plt.tight_layout()
# plt.savefig(f"{vis_save_to_root}/{region_name}_map.png")

- Calculate Country Aggregated data

In [ ]:
from RESource.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

all_cells_aggr = get_sub_nationally_aggregated_capacity(all_cells_gdf_plot,AGGREGATION_LEVEL)
aggregated_map = all_boundary_plot.merge(
    all_cells_aggr[
        [
            AGGREGATION_LEVEL,
            "potential_capacity_solar",
            "potential_capacity_wind",
            "Developable_area_solar",
            "Developable_area_wind",
            "geom_area_km2",
        ]
    ],
    on=AGGREGATION_LEVEL,
    how="left",
)


- Add calculated data

In [ ]:
aggregated_map['Availability_solar_pct']=aggregated_map['Developable_area_solar']/aggregated_map['geom_area_km2']*100
aggregated_map['Availability_wind_pct']=aggregated_map['Developable_area_wind']/aggregated_map['geom_area_km2']*100

- plot

In [ ]:
if aggregated_map.crs != plot_crs:
    aggregated_map_plot = aggregated_map.to_crs(plot_crs)
else:
    aggregated_map_plot = aggregated_map

- Plot combined Availability

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

all_cells_gdf_plot["LandAvailability_solar_pct"] = all_cells_gdf_plot["LandAvailability_ERA5_solar"] * 100
all_cells_gdf_plot["LandAvailability_wind_pct"]  = all_cells_gdf_plot["LandAvailability_ERA5_wind"]  * 100

fig, axes = plt.subplots(1, 2, figsize=(7, 5.5), dpi=300)  # extra height absorbed below

plot_specs = [
    ("LandAvailability_solar_pct", "Availability_solar_pct", "Solar", axes[0]),
    ("LandAvailability_wind_pct",  "Availability_wind_pct",  "Wind",  axes[1]),
]

cmap = mpl.colormaps.get_cmap("Greens")
norm = mpl.colors.Normalize(vmin=0, vmax=100)

# ── Set layout ONCE, before canvas.draw() ────────────────────────────────────
# bottom=0.15 reserves exactly enough room for one legend row + scenario box
fig.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.15, wspace=0.18)

for cell_col, agg_col, panel_title, ax in plot_specs:
    all_cells_gdf_plot.plot(
        column=cell_col, cmap=cmap, edgecolor="slategrey",
        linewidth=0.05, legend=False, vmin=0, vmax=100, ax=ax, alpha=1,
    )
    aggregated_map_plot.plot(
        color="none", edgecolor="#222222", linewidth=0.3, ax=ax, zorder=3,
    )
    ax, vre_legends = vis.get_existing_committed_VRE_plot(
        ax=ax,
        existing_VREs_gdf=existing_VREs_gdf,
        existing_VRE_type_column='Technology',
        committed_VREs_gdf=committed_VREs_gdf,
        committed_VRE_type_column='Technology',
        target_crs=CRS_m,
        marker_scale_existing=MARKER_SCALE_EXISTING/18,
        marker_scale_committed=MARKER_SCALE_COMMITTED/12,
        marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
        sites_legend_handle_scale=5,
    )
    for _, row in aggregated_map_plot.iterrows():
        point = row.geometry.representative_point()

    ax.set_title(f"{panel_title}  ({row[agg_col]:.1f}%)", fontsize=14, fontweight="bold", pad=6)
    ax.set_axis_off()

# ── Colorbar in inter-panel gap ───────────────────────────────────────────────
fig.canvas.draw()  # lock in positions — NO subplots_adjust after this

ax0_pos = axes[0].get_position()
ax1_pos = axes[1].get_position()

gap_center = (ax0_pos.x1 + ax1_pos.x0) / 2
cb_width   = 0.022
cb_height  = ax0_pos.height * 0.72
cb_left    = gap_center - cb_width / 2
cb_bottom  = ax0_pos.y0 + (ax0_pos.height - cb_height) / 2

cax = fig.add_axes([cb_left, cb_bottom, cb_width, cb_height])
sm  = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax, orientation="vertical")
cbar.set_label("Land availability (%)", fontsize=12, fontweight="bold", labelpad=8)
cbar.ax.tick_params(labelsize=11, width=2)
cbar.set_ticks([0, 20, 40, 60, 80, 100])
cbar.ax.yaxis.set_label_position("left")
cbar.ax.yaxis.tick_right()

# ── Legend + scenario — anchored to axes bottom, not figure bottom ────────────
# Using ax0_pos.y0 as reference makes these sit just below the maps
# regardless of what subplots_adjust does
BOTTOM_Y = ax0_pos.y0 - 0.07  # 10% below axes bottom edge, in figure fraction

leg = fig.legend(
    handles=vre_legends,
    fontsize=9,
    framealpha=0.92,
    edgecolor='#bbbbbb',
    fancybox=False,
    handlelength=1.4,
    handleheight=1.2,
    ncols=3,
    columnspacing=1.2,
    loc='lower left',
    bbox_to_anchor=(0.02, BOTTOM_Y),
    bbox_transform=fig.transFigure,
)

fig.text(
    0.95, BOTTOM_Y+0.01,
    "Scenario : BASELINE",
    ha='right', va='bottom',
    fontsize=9, color="#140202",
    transform=fig.transFigure,
    bbox=dict(boxstyle="round,pad=0.4", edgecolor="#bbbbbb", facecolor="lightgrey", alpha=0.8)
)

plt.savefig(
    f"{vis_save_to_root}/{region_code}_LandAvailability_solar_wind.tiff",
    bbox_inches="tight",   # fine now — legend is directly below axes, no orphan space
    transparent=False,
    dpi=300,
)
plt.show()

## Land Layers

### Canadian Landcover


In [ ]:
CAN_LC_raster_path=root/'data/downloaded_data/GAEZ/Rasters_in_use/LR/ter/slpmed30s.tif'
CAN_LC_raster_legends=pd.read_csv(root/'data/legends/LandCover_CANgov_2020_legend.csv')

In [ ]:
CAN_LC_raster_da = utils.get_raster_da(CAN_LC_raster_path) 

In [ ]:
# Ensure same CRS
if all_boundary_plot.crs != CAN_LC_raster_da.rio.crs:
    boundary_CAN_LC = all_boundary_plot.to_crs(CAN_LC_raster_da.rio.crs)
else:
    boundary_CAN_LC = all_boundary_plot
# Clip raster to boundary
CAN_LC_raster_da_clipped = CAN_LC_raster_da.rio.clip(boundary_CAN_LC.geometry, boundary_CAN_LC.crs)

- Review class distributions

In [ ]:
from RESource import lands
lands.plot_raster_class_distribution(CAN_LC_raster_da_clipped,
                                     CAN_LC_raster_legends,
                                     show=True,
                                     figsize=(8,3),
                                     save_path=vis_save_to_root/f'CAN_LC_class_distribution_{region_code}.png')


- Review layers mapping to existing VREs

In [ ]:
existing_VREs_gdf_with_terrains,terrains_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=CAN_LC_raster_da_clipped,
    legend_df=CAN_LC_raster_legends,
    class_col_name="CAN_LC",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=terrains_summary,
    class_col="CAN_LC_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Land Cover Class and Technology",
    figsize=(6, 2),
    wrap_width=25,
    fontsize=6,
    colors=["#3e1cd6", "#6f32e0"],
    save_to=vis_save_to_root/f"existing_VREs_by_land_cover_{region_code}.png"
)


In [ ]:
committed_VREs_gdf_with_terrains,terrains_summary = lands.assign_raster_class_to_points(
    gdf=committed_VREs_gdf,
    raster_da=CAN_LC_raster_da_clipped,
    legend_df=CAN_LC_raster_legends,
    class_col_name="CAN_LC",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=terrains_summary,
    class_col="CAN_LC_description",
    count_prefix="SiteCount_",
    title="Committed VRE Sites by Land Cover Class and Technology",
    figsize=(6, 2),
    wrap_width=25,
    fontsize=6,
    colors=["#d6851c", "#6f32e0"],
    save_to=vis_save_to_root/f"committed_VREs_by_land_cover_{region_code}.png"
)


- Define the layers to be plotted

In [ ]:
import numpy as np
layers_included:dict=cfg.get('custom_land_layers').get('rasters')[0]['class_inclusion']

# Unique CLC codes in your clipped area
unique_classes = np.unique(CAN_LC_raster_da_clipped.values[~np.isnan(CAN_LC_raster_da_clipped.values)])

layers_excluded = {
    tech: [c for c in unique_classes if c not in layers_included[tech] and c != 0]
    for tech in ['solar', 'wind']
}

In [ ]:
GAEZ_legend_anchor=(0.8, 0.8) # Tested to look clean

- Plot CAN Land Cover with defined layers and existing VREs

In [ ]:
classes_to_plot = layers_included # layers_included #None , plots all layers

if classes_to_plot is None:
    CAN_LC_save_to_path=vis_save_to_root/f"CAN_LC_{region_code}_allLayers.png"
    title = "CAN Land Cover - All Layers"
else:
# Update DOC contents (if needed)
    CAN_LC_save_to_path=vis_save_to_root/f"CAN_LC_{region_code}_forNewVREsites.png"
    title = "CAN Land Cover - For New VRE Sites"

fig2,ax2,save_to2=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=CAN_LC_raster_da_clipped,
    raster_legends=CAN_LC_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=all_boundary_plot,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=MARKER_SCALE_EXISTING,
    marker_scale_committed=MARKER_SCALE_COMMITTED,
    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
    legend_fontsize=10,
    title=title,
    output_path=CAN_LC_save_to_path,
    legend_anchor=GAEZ_legend_anchor,
)
utils.print_update(level=1,message=f"plot saved to: {CAN_LC_save_to_path}")

In [ ]:
classes_to_plot = None # layers_included #None , plots all layers

if classes_to_plot is None:
    CAN_LC_save_to_path=vis_save_to_root/f"CAN_LC_{region_code}_allLayers.png"
    title = "CAN Land Cover - All Layers"
else:
# Update DOC contents (if needed)
    CAN_LC_save_to_path=vis_save_to_root/f"CAN_LC_{region_code}_forNewVREsites.png"
    title = "CAN Land Cover - For New VRE Sites"

fig2,ax2,save_to2=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=CAN_LC_raster_da_clipped,
    raster_legends=CAN_LC_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=all_boundary_plot,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=MARKER_SCALE_EXISTING,
    marker_scale_committed=MARKER_SCALE_COMMITTED,
    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
    legend_fontsize=10,
    title=title,
    output_path=CAN_LC_save_to_path,
    legend_anchor=GAEZ_legend_anchor,
)
utils.print_update(level=1,message=f"plot saved to: {CAN_LC_save_to_path}")

### GAEZ

#### Terrain

In [ ]:
GAEZ_terrain_raster_path=root/'data/downloaded_data/GAEZ/Rasters_in_use/LR/ter/slpmed30s.tif'
GAEZ_terrain_raster_legends=pd.read_csv(root/'data/legends/gaez_terrains_legend.csv')
GAEZ_terrain_cfg= next((item for item in cfg.get('GAEZ').get('raster_types') if 'terrain_resources' in item.get('name', '')), None)

- Load Raster Data as data-array

In [ ]:
GAEZ_terrain_raster_da = utils.get_raster_da(GAEZ_terrain_raster_path) 

- Clip Raster to boundary (otherwise raster distribution will show wrong numbers)

In [ ]:
# Ensure same CRS
if all_boundary_plot.crs != GAEZ_terrain_raster_da.rio.crs:
    boundary_GAEZ = all_boundary_plot.to_crs(GAEZ_terrain_raster_da.rio.crs)
else:
    boundary_GAEZ = all_boundary_plot
# Clip raster to boundary
GAEZ_terrain_raster_da_clipped = GAEZ_terrain_raster_da.rio.clip(boundary_GAEZ.geometry, boundary_GAEZ.crs)

In [ ]:
# from RESource import lands
# GAEZ_terrain_raster_da_clipped=utils.get_raster_da(lands.clip_to_boundary_and_resample_raster(in_raster_config= GAEZ_terrain_cfg,
#                                            boundary_name=region_name,
#                                            boundary=all_boundary_plot,
#                                            CRS_meters=CRS_m,
#                                            source_raster_path=GAEZ_terrain_raster_path,
#                                            ))

- Review class distributions

In [ ]:
from RESource import lands
lands.plot_raster_class_distribution(GAEZ_terrain_raster_da_clipped,
                                     GAEZ_terrain_raster_legends,
                                     show=True,
                                     figsize=(8,3),
                                     save_path=vis_save_to_root/f'GAEZ_terrains_class_distribution_{region_code}.png')


- Review layers mapping to existing VREs

In [ ]:
existing_VREs_gdf_with_terrains,terrains_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=GAEZ_terrain_raster_da_clipped,
    legend_df=GAEZ_terrain_raster_legends,
    class_col_name="GAEZ_terrain",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=terrains_summary,
    class_col="GAEZ_terrain_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Terrain Class and Technology",
    figsize=(6, 2),
    wrap_width=25,
    fontsize=6,
    colors=["#3e1cd6", "#6f32e0"],
    save_to=vis_save_to_root/f"existing_VREs_by_terrain_{region_code}.png"
)


In [ ]:
committed_VREs_gdf_with_terrains,terrains_summary = lands.assign_raster_class_to_points(
    gdf=committed_VREs_gdf,
    raster_da=GAEZ_terrain_raster_da_clipped,
    legend_df=GAEZ_terrain_raster_legends,
    class_col_name="GAEZ_terrain",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=terrains_summary,
    class_col="GAEZ_terrain_description",
    count_prefix="SiteCount_",
    title="Committed VRE Sites by Terrain Class and Technology",
    figsize=(6, 2),
    wrap_width=25,
    fontsize=6,
    colors=["#d6851c", "#6f32e0"],
    save_to=vis_save_to_root/f"committed_VREs_by_terrain_{region_code}.png"
)


- Define the layers to be plotted

In [ ]:
import numpy as np
layers_excluded:dict=GAEZ_terrain_cfg['class_exclusion']

# Unique CLC codes in your clipped area
unique_classes = np.unique(GAEZ_terrain_raster_da_clipped.values[~np.isnan(GAEZ_terrain_raster_da_clipped.values)])

layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}

- Plot GAEZ Terrain with defined layers and existing VREs

In [ ]:
classes_to_plot = layers_included # layers_included #None , plots all layers

if classes_to_plot is None:
    GAEZ_terrain_save_to_path=vis_save_to_root/f"GAEZ_terrains_{region_code}_allLayers.png"
    title = "GAEZ Terrains - All Layers"
else:
# Update DOC contents (if needed)
    GAEZ_terrain_save_to_path=vis_save_to_root/f"GAEZ_terrains_{region_code}_forNewVREsites.png"
    title = "GAEZ Terrains - For New VRE Sites"

fig2,ax2,save_to2=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=GAEZ_terrain_raster_da_clipped,
    raster_legends=GAEZ_terrain_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=all_boundary_plot,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=MARKER_SCALE_EXISTING,
    marker_scale_committed=MARKER_SCALE_COMMITTED,
    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
    legend_fontsize=10,
    title=title,
    output_path=GAEZ_terrain_save_to_path,
    legend_anchor=GAEZ_legend_anchor,
)
utils.print_update(level=1,message=f"plot saved to: {GAEZ_terrain_save_to_path}")

In [ ]:
classes_to_plot = None # layers_included #None , plots all layers

if classes_to_plot is None:
    GAEZ_terrain_save_to_path=vis_save_to_root/f"GAEZ_terrains_{region_code}_allLayers.png"
    title = "GAEZ Terrains - All Layers"
else:
# Update DOC contents (if needed)
    GAEZ_terrain_save_to_path=vis_save_to_root/f"GAEZ_terrains_{region_code}_forNewVREsites.png"
    title = "GAEZ Terrains - For New VRE Sites"

fig2,ax2,save_to2=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=GAEZ_terrain_raster_da_clipped,
    raster_legends=GAEZ_terrain_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=all_boundary_plot,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=MARKER_SCALE_EXISTING,
    marker_scale_committed=MARKER_SCALE_COMMITTED,
    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
    legend_fontsize=10,
    title=title,
    output_path=GAEZ_terrain_save_to_path,
    legend_anchor=GAEZ_legend_anchor,
)
utils.print_update(level=1,message=f"plot saved to: {GAEZ_terrain_save_to_path}")

### Exclusion

In [ ]:
GAEZ_exclusion_raster_path = root/"data/downloaded_data/GAEZ/Rasters_in_use/LR/excl/exclusion_2017.tif"
GAEZ_exclusion_raster_legends=pd.read_csv(root/"data/legends/gaez_exclusion_legend.csv")
GAEZ_exclusion_cover_cfg= next((item for item in cfg.get('GAEZ').get('raster_types') if 'exclusion_areas' in item.get('name', '')), None)

- Load Raster Data as data-array

In [ ]:
GAEZ_exclusion_raster_data = utils.get_raster_da(GAEZ_exclusion_raster_path)

- Clip to WB6 Boundary

In [ ]:
# Ensure same CRS
if all_boundary_plot.crs != GAEZ_exclusion_raster_data.rio.crs:
    boundary_dissolved = all_boundary_plot.to_crs(GAEZ_exclusion_raster_data.rio.crs)

# Clip raster to boundary
GAEZ_exclusion_raster_data_clipped = GAEZ_exclusion_raster_data.rio.clip(boundary_dissolved.geometry, boundary_dissolved.crs)

- Plot Class Distribution

In [ ]:
lands.plot_raster_class_distribution(GAEZ_exclusion_raster_data_clipped,
                                     GAEZ_exclusion_raster_legends,
                                     show=True,
                                     figsize=(8,3),
                                     save_path=vis_save_to_root/f'GAEZ_Exclusion_class_distribution_{region_code}.png')

- Review existing sites and excluded lands

In [ ]:
existing_VREs_gdf_with_exclusions,exclusions_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=GAEZ_exclusion_raster_data_clipped,
    legend_df=GAEZ_exclusion_raster_legends,
    class_col_name="GAEZ_exclusion",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=exclusions_summary,
    class_col="GAEZ_exclusion_description",
    count_prefix="SiteCount_",
    title="Existing VRE Sites by Exclusion Class and Technology",
    figsize=(8, 2),
    wrap_width=25,
    fontsize=8,
    colors=["#6f32e0"],
    save_to=vis_save_to_root/f"existing_VREs_by_exclusions_{region_code}.png"
)

In [ ]:
committed_VREs_gdf_with_exclusions,exclusions_summary = lands.assign_raster_class_to_points(
    gdf=existing_VREs_gdf,
    raster_da=GAEZ_exclusion_raster_data_clipped,
    legend_df=GAEZ_exclusion_raster_legends,
    class_col_name="GAEZ_exclusion",
    resource_type_col_name='Technology'
)

vis.plot_vre_sites_by_landcover(
    df=exclusions_summary,
    class_col="GAEZ_exclusion_description",
    count_prefix="SiteCount_",
    title="Committed VRE Sites by Exclusion Class and Technology",
    figsize=(8, 2),
    wrap_width=25,
    fontsize=8,
    colors=[ "#6f32e0"],
    save_to=vis_save_to_root/f"committed_VREs_by_exclusions_{region_code}.png"
)

- Define layers to be plotted

In [ ]:
layers_excluded:dict=GAEZ_exclusion_cover_cfg['class_exclusion']

# Unique GAEZ terrain codes in your clipped area
unique_classes = np.unique(GAEZ_exclusion_raster_data.values[~np.isnan(GAEZ_exclusion_raster_data.values)])

layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}

- Plot

In [ ]:
classes_to_plot = layers_included  #None , plots all layers

if classes_to_plot is None:
    GAEZ_exclusion_save_to_path=vis_save_to_root/f"GAEZ_exclusion_{region_code}_allLayers.png"
    title = "GAEZ Exclusions - All Layers"
else:
# Update DOC contents (if needed)
    GAEZ_exclusion_save_to_path=vis_save_to_root/f"GAEZ_exclusion_{region_code}_forNewVREsites.png"
    title = "GAEZ Exclusions - For New VRE Sites"

fig3,ax3,save_to3=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=GAEZ_exclusion_raster_data_clipped,
    raster_legends=GAEZ_exclusion_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=all_boundary_plot,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=MARKER_SCALE_EXISTING,
    marker_scale_committed=MARKER_SCALE_COMMITTED,
    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
    legend_fontsize=10,
    title=title,
    output_path=GAEZ_exclusion_save_to_path,
    legend_anchor=GAEZ_legend_anchor,

)
utils.print_update(level=1,message=f"plot saved to: {GAEZ_exclusion_save_to_path}")

In [ ]:
classes_to_plot = None  #None , plots all layers

if classes_to_plot is None:
    GAEZ_exclusion_save_to_path=vis_save_to_root/f"GAEZ_exclusion_{region_code}_allLayers.png"
    title = "GAEZ Exclusions - All Layers"
else:
# Update DOC contents (if needed)
    GAEZ_exclusion_save_to_path=vis_save_to_root/f"GAEZ_exclusion_{region_code}_forNewVREsites.png"
    title = "GAEZ Exclusions - For New VRE Sites"

fig3,ax3,save_to3=vis.plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=GAEZ_exclusion_raster_data_clipped,
    raster_legends=GAEZ_exclusion_raster_legends,
    classes_to_plot=classes_to_plot, #layers_included,                 # <- all classes
    boundary=all_boundary_plot,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    vre_type_column="Technology",
    marker_scale_existing=MARKER_SCALE_EXISTING,
    marker_scale_committed=MARKER_SCALE_COMMITTED,
    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
    legend_fontsize=10,
    title=title,
    output_path=GAEZ_exclusion_save_to_path,
    legend_anchor=GAEZ_legend_anchor,

)
utils.print_update(level=1,message=f"plot saved to: {GAEZ_exclusion_save_to_path}")

# Grid

In [ ]:
# all_lines=[combined_store[region]['lines'] for region in combined_store]
# # all_lines = gpd.GeoDataFrame(pd.concat(all_lines, ignore_index=True), crs=all_lines[0].crs)

In [ ]:
# # Convert to numeric
# all_lines['voltage_kv'] = pd.to_numeric(all_lines['voltage'], errors='coerce') / 1000

# # Define voltage bins
# bins = [0, 12, 25, 132, 220, float("inf")]
# labels = ["<12 kV", "12–25 kV", "25–132 kV", "132–220 kV", "≥220 kV"]
# all_lines['voltage_class'] = pd.cut(all_lines['voltage_kv'], bins=bins, labels=labels, right=False)


- Plot gird

In [ ]:
# from matplotlib.patches import Patch
# ax=all_boundary_gdf.plot(edgecolor='black', facecolor='grey', linewidth=0.2, figsize=(10, 8),alpha=0.1)

# ax.set_axis_off()

# if existing_VREs_gdf is not None and not existing_VREs_gdf.empty:

#     existing_VREs_gdf[existing_VREs_gdf['Technology'].str.lower()=='wind'].plot(ax=ax, color='None', edgecolor='blue',markersize=60, linewidth=0.8,marker='o', label='Existing VREs',alpha=1,zorder=2)

#     # Add legend for existing wind (purple)
#     legend_handles = [Patch(facecolor='none', edgecolor='purple', label='Existing Wind', alpha=1)]
#     ax.legend(handles=legend_handles, loc='upper left', fontsize=8, frameon=False)

#     existing_VREs_gdf[existing_VREs_gdf['Technology'].str.lower()=='solar'].plot(ax=ax, color='None', edgecolor='orangered',markersize=60,linewidth=0.8, marker='o', label='Existing VREs',alpha=1,zorder=2)
# all_lines.plot('voltage_class',ax=ax, figsize=(10, 8), legend=True)

# plt.savefig(vis_save_to_root/f"WB6_existing_VREs_and_lines_{country_kwd}.png", dpi=500, bbox_inches='tight')

# Population Density vs Cells

In [ ]:
import os
import glob
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import geopandas as gpd

# ── Paths ─────────────────────────────────────────────────────────────────────
WORLDPOP_DIR = root/"data/downloaded_data/worldpop/processed"

# ── Load all rasters ──────────────────────────────────────────────────────────
raster_paths = sorted(glob.glob(os.path.join(WORLDPOP_DIR, "*_pd_clipped.tif")))

rasters = {}
for path in raster_paths:
    iso3= os.path.basename(path).split("_")[0]   # AL, BA, ME, MK, RS, XK
    if iso3 =='CAN'.lower():
        with rasterio.open(path) as src:
            data   = src.read(1).astype(np.float64)
            bounds = src.bounds
            crs    = src.crs
        data[data <= 0] = np.nan
        rasters[iso3] = {"data": data, "bounds": bounds, "crs": crs, "path": path}

print(f"Loaded {len(rasters)} rasters: {list(rasters.keys())}")

In [ ]:
TARGET_YEAR=2020
# ── Class scheme (reuse from before) ─────────────────────────────────────────
CLASS_BINS   = [0, 1, 5, 25, 100, 500, 2000, np.inf]
CLASS_COLORS = ["#F7FCF0","#CCEBC5","#7BCCC4","#4EB3D3",
                "#2B8CBE","#0868AC","#084081"]
CLASS_LABELS = ["<1","1–5","5–25","25–100","100–500","500–2k",">2k"]

CMAP_DISC = mcolors.ListedColormap(CLASS_COLORS)
NORM_DISC = mcolors.BoundaryNorm(np.arange(0.5, len(CLASS_COLORS)+1.5), CMAP_DISC.N)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import rasterio
from rasterio.crs import CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.merge import merge as rmerge
import geopandas as gpd
from shapely.geometry import box as shapely_box

# ── Class scheme ──────────────────────────────────────────────────────────────
CLASS_BINS   = [0, 1, 5, 25, 100, 500, 2000, np.inf]
CLASS_COLORS = ["#F7FCF0","#CCEBC5","#7BCCC4","#4EB3D3",
                "#2B8CBE","#0868AC","#084081"]
CLASS_LABELS = ["<1","1–5","5–25","25–100","100–500","500–2k",">2k"]

CMAP_DISC = mcolors.ListedColormap(CLASS_COLORS)
NORM_DISC = mcolors.BoundaryNorm(np.arange(0.5, len(CLASS_COLORS)+1.5), CMAP_DISC.N)
CMAP_DISC.set_bad("#ffffff")

def classify(arr):
    out = np.full_like(arr, np.nan, dtype=np.float64)
    for k, (lo, hi) in enumerate(zip(CLASS_BINS[:-1], CLASS_BINS[1:]), 1):
        out[(arr >= lo) & (arr < hi)] = k
    return out

# ── Target CRS — everything reprojects to this ────────────────────────────────
TARGET_CRS = CRS_m   # BC Albers; swap to 4326 for geographic degrees

# ── Merge rasters — capture native CRS before closing ─────────────────────────
src_files  = [rasterio.open(r["path"]) for r in rasters.values()]
src_crs    = src_files[0].crs
mosaic, mosaic_tf = rmerge(src_files, nodata=np.nan)
for s in src_files:
    s.close()

data = mosaic[0].astype(np.float64)
data[data <= 0] = np.nan

rows, cols = data.shape

# ── Warp raster to TARGET_CRS ─────────────────────────────────────────────────
dst_transform, dst_width, dst_height = calculate_default_transform(
    src_crs, TARGET_CRS, cols, rows,
    left   = mosaic_tf.c,
    bottom = mosaic_tf.f + mosaic_tf.e * rows,   # south
    right  = mosaic_tf.c + mosaic_tf.a * cols,   # east
    top    = mosaic_tf.f,                         # north
)

data_warped = np.full((dst_height, dst_width), np.nan, dtype=np.float64)
reproject(
    source       = data,
    destination  = data_warped,
    src_transform= mosaic_tf,
    src_crs      = src_crs,
    dst_transform= dst_transform,
    dst_crs      = TARGET_CRS,
    resampling   = Resampling.nearest,
    src_nodata   = np.nan,
    dst_nodata   = np.nan,
)

# ── Recompute extent from warped raster ───────────────────────────────────────
west  = dst_transform.c
north = dst_transform.f
east  = west  + dst_transform.a * dst_width
south = north + dst_transform.e * dst_height
data  = data_warped                              # use warped array from here on

# ── Reproject overlays to TARGET_CRS ─────────────────────────────────────────
if all_cells_gdf_plot.crs != TARGET_CRS:
    all_cells_gdf_plot = all_cells_gdf_plot.to_crs(TARGET_CRS)

if all_boundary_plot.crs != TARGET_CRS:
    all_boundary_plot = all_boundary_plot.to_crs(TARGET_CRS)

# ── Clip to mosaic extent ─────────────────────────────────────────────────────
roi           = shapely_box(west, south, east, north)
boundary_clip = all_boundary_plot[all_boundary_plot.intersects(roi)]
cells_clip    = all_cells_gdf_plot.cx[west:east, south:north]

# ── Diagnostics ───────────────────────────────────────────────────────────────
print("target CRS             :", TARGET_CRS)
print("warped raster extent   :", west, south, east, north)
print("cells bbox (reprojected):", all_cells_gdf_plot.total_bounds)
print("boundary in extent     :", len(boundary_clip))
print("cells in extent        :", len(cells_clip))

In [ ]:
# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8), dpi=1000)
ax.set_facecolor("None")

fig.patch.set_facecolor('white')
# ── Class scheme ──────────────────────────────────────────────────────────────
CLASS_BINS   = [0, 1, 5, 25, 100, 500, 2000, np.inf]
CLASS_LABELS = ["<1", "1–5", "5–25", "25–100", "100–500", "500–2k", ">2k"]
N = len(CLASS_LABELS)

# ── Pick one — comment out the others ────────────────────────────────────────
BASE_CMAP = plt.get_cmap("YlOrRd",   N)   # light yellow → deep red  (population classic)
# BASE_CMAP = plt.get_cmap("YlGnBu", N)   # yellow → green → blue    (cool-to-warm feel)
# BASE_CMAP = plt.get_cmap("plasma",  N)   # purple → orange → yellow (perceptually uniform)
# BASE_CMAP = plt.get_cmap("BuPu",    N)   # white-blue → deep purple

# Colours extracted from the base — legend and imshow share the same source
CLASS_COLORS = [mcolors.to_hex(BASE_CMAP(i)) for i in range(N)]

CMAP_DISC = mcolors.ListedColormap(CLASS_COLORS)
NORM_DISC = mcolors.BoundaryNorm(np.arange(0.5, N + 1.5), CMAP_DISC.N)
CMAP_DISC.set_bad("#f5f5f5")   # nodata → light grey (more neutral than white)

ax.imshow(
    classify(data),
    extent=[west, east, south, north],
    cmap=CMAP_DISC,
    norm=NORM_DISC,
    interpolation='nearest',
    origin='upper',
    zorder=1,
)
# ── Overlays ──────────────────────────────────────────────────────────────────
if len(cells_clip) > 0:
    cells_clip.plot(
        ax=ax,
        facecolor='none',
        edgecolor="#413d39",
        linewidth=0.2,
        linestyle=':',
        zorder=4,
    )
else:
    print("WARNING: cells_clip is empty — CRS or extent mismatch not resolved")

boundary_clip.boundary.plot(
    ax=ax, edgecolor='#1a1a1a', linewidth=0.2, zorder=5,
)

# ── Country labels ────────────────────────────────────────────────────────────
stroke = [pe.withStroke(linewidth=2.5, foreground='white')]
# label_positions = {
#     'BC': (-124.5, 54.5, 'British Columbia'),
# }
# for iso2, (lx, ly, lname) in label_positions.items():
#     ax.text(lx, ly, lname,
#             fontsize=8, ha='center', va='center',
#             fontweight='bold', color='#1a1a1a',
#             path_effects=stroke, zorder=6)

# ── VRE overlay (one call — solar and wind are both in existing_VREs_gdf) ─────
ax, vre_legends = vis.get_existing_committed_VRE_plot(
    ax=ax,
    existing_VREs_gdf=existing_VREs_gdf,
    existing_VRE_type_column='Technology',
    committed_VREs_gdf=committed_VREs_gdf,
    committed_VRE_type_column='Technology',
    target_crs=CRS_m,        # ← must match raster, not a hardcoded CRS_d
    marker_scale_existing=MARKER_SCALE_EXISTING-1.2,
    marker_scale_committed=MARKER_SCALE_COMMITTED-0.2,
    marker_highlight_width=MARKER_HIGHLIGHT_WIDTH,
)
# ── Legend ────────────────────────────────────────────────────────────────────
# ── Legend — derived from same colormap, guaranteed in sync ───────────────────
pop_patches = [
    mpatches.Patch(facecolor=CLASS_COLORS[i], edgecolor='#888',
                   linewidth=0.4, label=CLASS_LABELS[i])
    for i in range(N)
]
line_patches = [
    Line2D([0],[0], color='#413d39', lw=1.2, ls=':', label='ERA5 cells'),
    Line2D([0],[0], color='#1a1a1a', lw=1.4,         label='Boundary'),
]
all_handles = pop_patches + line_patches + vre_legends

leg = ax.legend(
    handles=all_handles,
    title='People / km²',
    title_fontsize=10,
    fontsize=9,
    loc='lower left',
    framealpha=0.92,
    edgecolor='#bbbbbb',
    fancybox=False,
    handlelength=1.4,
    handleheight=1.2,
)
leg.get_title().set_fontweight('bold')

ax.set_title(f'{region_code} Population Density {TARGET_YEAR}',
             fontsize=14, fontweight='bold', pad=10)
ax.axis('off')


plt.tight_layout()
out_fig = vis_save_to_root / f'{region_code}_population_density_{TARGET_YEAR}.png'
fig.savefig(out_fig, dpi=500, transparent=True, facecolor='None')
plt.show()
print(f'Saved → {out_fig}')